## __Phase 4c: Baseline bias experiments (NN Time-weighted vs. PHMM)__

`04_evaluation.ipynb` / `run_phase4_evaluation.py` runs four experiments testing *why*
the PHMM Forward alignment misclassifies RF signals: track length, intra-track AIS
sampling density (time and distance), vessel stationarity, and point-level time/distance
proximity to the nearest AIS fix. This compares the same four experiments for the NN
Time-weighted baseline against a freshly recomputed PHMM reference, on the same held-out
validation split, so the two methods are compared on identical RF points.

The full experiment run (all six boxplot/t-test pairs and the summary CSV) now lives in
`scripts/evaluation/run_phase4c_bias_experiments.py`:

```bash
python -m scripts.evaluation.run_phase4c_bias_experiments --error-model uniform
```

This notebook keeps a light interactive version for exploration only: it imports that
script's own data-loading and split-building functions (nothing duplicated here) and
plots two of the six experiments live via `scripts.evaluation.plots`.

In [ ]:
import matplotlib.pyplot as plt

from scripts.evaluation import plots
from scripts.evaluation.run_phase4c_bias_experiments import (
    build_validation_split,
    enrich,
    load_data,
    nn_time_weighted_splits,
    phmm_reference_splits,
)

In [ ]:
# Select which RF bearing-error model's data to evaluate (must match the
# --error-model used to generate/run the upstream scripts)
ERROR_MODEL = "uniform"  # "uniform" or "gaussian"
DATA_DIR = "../data/processed" if ERROR_MODEL == "uniform" else f"../data/processed/{ERROR_MODEL}"

data = load_data(DATA_DIR, base_dir="../data/processed")
split = build_validation_split(data)
print("RF points in the shared validation split:", split["n_val"])

In [ ]:
splits_phmm = enrich(*phmm_reference_splits(data, split), data, add_trajectory=False)
splits_nn = enrich(*nn_time_weighted_splits(data, split), data, add_trajectory=True)

### __Experiment 1: track length (#points)__

In [ ]:
plots.boxplot_correct_chosen_shouldbe_compare(
    splits_phmm, splits_nn, "#points", "Number of AIS messages per trajectory",
    "Experiment 1: track length, Correct vs Chosen vs Should-be",
    names=("PHMM Forward (corrected)", "NN Time-weighted"), fmt="{:.1f}",
)
plt.show()

### __Experiment 4b: distance to nearest AIS fix__

In [ ]:
plots.boxplot_correct_chosen_shouldbe_compare(
    splits_phmm, splits_nn, "distance_diff", "Kilometers",
    "Experiment 4b: distance to nearest AIS fix, Correct vs Chosen vs Should-be",
    names=("PHMM Forward (corrected)", "NN Time-weighted"), fmt="{:.3f}",
)
plt.show()